# Phase 3 — Feature engineering

# Setting up libraries

In [1]:
import pandas as pd
import numpy as np

import datetime as dt
from datetime import date

from dateutil.relativedelta import relativedelta

# Load dataset

In [ ]:
print("table: online retail transactions")
retail_data = pd.read_csv('../data/interim/cleaned_retail_transactions.csv')
display(retail_data.head())

table: online retail transactions


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_cancellation,is_non_product,is_missing_customer,is_customer_cancellation,is_stock_adjustment
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,False,False,False


## Basic information of the table

In [ ]:
retail_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1028761 entries, 0 to 1028760
Data columns (total 13 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   invoice                   1028761 non-null  str    
 1   stockcode                 1028761 non-null  str    
 2   description               1028761 non-null  str    
 3   quantity                  1028761 non-null  int64  
 4   invoicedate               1028761 non-null  str    
 5   price                     1028761 non-null  float64
 6   customer_id               797885 non-null   float64
 7   country                   1028761 non-null  str    
 8   is_cancellation           1028761 non-null  bool   
 9   is_non_product            1028761 non-null  bool   
 10  is_missing_customer       1028761 non-null  bool   
 11  is_customer_cancellation  1028761 non-null  bool   
 12  is_stock_adjustment       1028761 non-null  bool   
dtypes: bool(5), float64(2), int64(1), str(

## Creating a copy of the table as transaction table

In [ ]:
transaction_tbl = retail_data.copy(deep= False)

## Changing the data types

In [7]:
transaction_tbl["customer_id"] = transaction_tbl["customer_id"].astype("Int64")

In [8]:
transaction_tbl["invoicedate"] = pd.to_datetime(transaction_tbl["invoicedate"], errors='coerce')

## Checking the table again to verify for the data types

In [9]:
transaction_tbl.info()

<class 'pandas.DataFrame'>
RangeIndex: 1028761 entries, 0 to 1028760
Data columns (total 13 columns):
 #   Column                    Non-Null Count    Dtype         
---  ------                    --------------    -----         
 0   invoice                   1028761 non-null  str           
 1   stockcode                 1028761 non-null  str           
 2   description               1028761 non-null  str           
 3   quantity                  1028761 non-null  int64         
 4   invoicedate               1028761 non-null  datetime64[us]
 5   price                     1028761 non-null  float64       
 6   customer_id               797885 non-null   Int64         
 7   country                   1028761 non-null  str           
 8   is_cancellation           1028761 non-null  bool          
 9   is_non_product            1028761 non-null  bool          
 10  is_missing_customer       1028761 non-null  bool          
 11  is_customer_cancellation  1028761 non-null  bool          
 1

In [ ]:
# Revenue calculation
transaction_tbl["revenue"] = transaction_tbl["quantity"] * transaction_tbl["price"]

In [10]:
# Extract year component
transaction_tbl["invoice_year"] = transaction_tbl["invoicedate"].dt.year

# Extract month component
transaction_tbl["invoice_month"] = transaction_tbl["invoicedate"].dt.month

# Extract year-month component
transaction_tbl["year_month"] = transaction_tbl["invoicedate"].dt.to_period("M")

# Extract quarter component
transaction_tbl["invoice_quarter"] = transaction_tbl["invoicedate"].dt.quarter

# Extract weekend indicator
transaction_tbl["is_weekend"] = transaction_tbl["invoicedate"].dt.dayofweek >= 5

# Extract day of month component
transaction_tbl["invoice_day"] = transaction_tbl["invoicedate"].dt.day

# Extract hour component
transaction_tbl["invoice_hour"] = transaction_tbl["invoicedate"].dt.hour

# Extract day of week component
transaction_tbl["invoice_weekday"] = transaction_tbl["invoicedate"].dt.weekday

In [11]:
# Create a new column to indicate if the quantity is negative
transaction_tbl["is_negative_quantity"] = (transaction_tbl["quantity"] < 0)

In [12]:
# Create a new column to indicate valid sales
transaction_tbl["is_valid_sale"] = ((transaction_tbl["is_cancellation"] == False) &
    (transaction_tbl["quantity"] > 0) &
    (transaction_tbl["price"] > 0))

In [ ]:
order_summary = (
    transaction_tbl[transaction_tbl["is_valid_sale"] == True]
    .groupby("invoice")
    .agg(
        order_revenue=("revenue", "sum"),
        order_units=("quantity", "sum"),
        order_product_count=("stockcode", "nunique"),
        order_line_count=("stockcode", "size")
    )
    .reset_index()
)